In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
# import pybedtools
import intervaltree  # Or use GenomeIntervalTree from intervaltree-bio
from tqdm import tqdm
import pandas as pd
from kneed import KneeLocator

In [ ]:
chr_colors = {
    'chr1': '#FF6B6B', 'chr2': '#4ECDC4', 'chr3': '#45B7D1', 'chr4': '#FFA07A',
    'chr5': '#92D050', 'chr6': '#D35FB7', 'chr7': '#FFC000', 'chr8': '#00B0F0',
    'chr9': '#A2D96C', 'chr10': '#C00000', 'chr11': '#7030A0', 'chr12': '#FF5733',
    'chr13': '#00AEEF', 'chr14': '#FF99CC', 'chr15': '#8FD8D8', 'chr16': '#F6546A',
    'chr17': '#468499', 'chr18': '#FFD700', 'chr19': '#088DA5', 'chr20': '#F08080',
    'chr21': '#6A5ACD', 'chr22': '#65B891', 'chr23': '#FFA500', 'chr24': '#BA55D3',
    'chr25': '#9370DB', 'chr26': '#3CB371', 'chr27': '#7B68EE', 'chr28': '#40E0D0',
    'chrX': '#FF1493',  # Distinct pink for X
    'chrY': '#14FF82'   # Dark blue for Y
}

In [ ]:
dup = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/genome-annotation/biser/hifiasm-041425/segdup_output_duplicateLinkRemoved.bedpe",
    sep="\t",
    header=None,
    names=["chr1", "start1", "end1", "chr2", "start2", "end2","reference","score","strand1","strand2","max_len","aln_len","cigar","optional"]
)
print(dup.shape)
# Assuming 'dup' is your DataFrame
dup = pd.concat([
    dup[['chr1', 'start1', 'end1']].rename(columns={'chr1': 'chrom', 'start1': 'start', 'end1': 'end'}),
    dup[['chr2', 'start2', 'end2']].rename(columns={'chr2': 'chrom', 'start2': 'start', 'end2': 'end'})
], ignore_index=True)

In [ ]:
## Total number of basepairs 
def merge_segdup_intervals(df):
    """
    Merge overlapping or adjacent segdup hotspot intervals
    """
    merged_data = []
    
    # Group by chromosome
    for chrom, group in df.groupby('chrom'):
        # Sort by start position
        sorted_group = group.sort_values('start')
        
        if len(sorted_group) == 0:
            continue
            
        merged_intervals = []
        current_start = sorted_group.iloc[0]['start']
        current_end = sorted_group.iloc[0]['end']
        
        for i in range(1, len(sorted_group)):
            row = sorted_group.iloc[i]
            start, end = row['start'], row['end']
            
            # Check for overlap or adjacency
            if start <= current_end:
                # Overlapping or adjacent - extend current interval if needed
                current_end = max(current_end, end)
            else:
                # No overlap - save current interval and start new one
                merged_intervals.append({
                    'chrom': chrom,
                    'start': current_start,
                    'end': current_end,
                    'Length': current_end - current_start
                })
                current_start = start
                current_end = end
        
        # Don't forget the last interval
        merged_intervals.append({
            'chrom': chrom,
            'start': current_start,
            'end': current_end,
            'Length': current_end - current_start
        })
        
        merged_data.extend(merged_intervals)
    
    return pd.DataFrame(merged_data)

In [ ]:
# hotspot_df = dup
# Merge the hotspot intervals
merged_segdup = merge_segdup_intervals(dup)
print(merged_segdup.shape)

In [ ]:
merged_segdup["Length"].sum()

In [ ]:
# Load genes with overlap information 
## This is calculated from calculate_segdup_geneOverlap_newHiFiAnnotation.ipynb
genes = pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/segdup-investigation/hifiasm_gene_segDup_overlapInfo_092525.tsv", sep="\t")
genes["Start"] = genes["Start"].astype(int)
genes["End"] = genes["End"].astype(int)
genes["overlaps_dup"] = genes["overlaps_dup"].astype(bool)
genes

In [ ]:
repeats = pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/output/outputs-from-repeatmasker/hifiasm-041425-haphic/assembly_final.sorted.headerRenamed.fasta.gff3",
                                       sep='\t', 
                 skiprows=1, 
                 header=None,
                 names=['seqid', 'source', 'type', 'Start', 'End', 'score', 
                       'strand', 'phase', 'attributes'])

seq_to_chr = {
    "seq1": "chrX",
    "seq23": "chrY",
    "seq2": "chr1",
    "seq3": "chr2",
    "seq4": "chr3",
    "seq5": "chr4",
    "seq6": "chr5",
    "seq7": "chr6",
    "seq8": "chr7",
    "seq9": "chr8",
    "seq10": "chr9",
    "seq11": "chr10",
    "seq12": "chr11",
    "seq13": "chr12",
    "seq14": "chr13",
    "seq15": "chr14",
    "seq16": "chr15",
    "seq17": "chr16",
    "seq18": "chr17",
    "seq19": "chr18",
    "seq20": "chr19",
    "seq21": "chr20",
    "seq22": "chr21",
    "seq24": "chr22",
    "seq25": "chr23",
    "seq26": "chr24",
    "seq27": "chr25",
    "seq28": "chr26",
    "seq29": "chr27",
    "seq30": "chr28"
}
# Apply the chromosome mapping to the 'seqid' column
repeats["Chromosome"] = repeats["seqid"].replace(seq_to_chr)

# Display the dataframe
repeats

In [ ]:

def categorize_repeats(repeat_df):
    """
    Categorize RepeatMasker repeats into major classes based on attributes
    """
    # Create masks for each category
    # Unknown/Unclassified
    unknown_mask = (
        repeat_df['attributes'].str.contains('Unknown', na=False) |
        repeat_df['attributes'].str.contains('Unclassified', na=False) |
        repeat_df['attributes'].str.contains('rnd-', na=False)  # Often unclassified repeats
    )
    
    # LTR elements
    ltr_mask = (
        repeat_df['attributes'].str.contains('LTR', na=False) |
        repeat_df['attributes'].str.contains('ltr-', na=False) |
        repeat_df['attributes'].str.contains('Retroviral', na=False) |
        repeat_df['attributes'].str.contains('Ty1/Copia', na=False) |
        repeat_df['attributes'].str.contains('Gypsy/DIRS1', na=False) |
        repeat_df['attributes'].str.contains('BEL/Pao', na=False)
    )
    
    # SINE elements
    sine_mask = (
        repeat_df['attributes'].str.contains('SINE', na=False) |
        repeat_df['attributes'].str.contains('Alu', na=False)  # Common SINE family
    )
    
    # LINE elements
    line_mask = (
        repeat_df['attributes'].str.contains('LINE', na=False) |
        repeat_df['attributes'].str.contains('L1', na=False) |
        repeat_df['attributes'].str.contains('L2', na=False) |
        repeat_df['attributes'].str.contains('CR1', na=False) |
        repeat_df['attributes'].str.contains('Rex', na=False) |
        repeat_df['attributes'].str.contains('RTE', na=False) |
        repeat_df['attributes'].str.contains('CIN4', na=False)
    )
    
    # DNA transposons
    dna_transposon_mask = (
        repeat_df['attributes'].str.contains('DNA', na=False) |
        repeat_df['attributes'].str.contains('hobo-Activator', na=False) |
        repeat_df['attributes'].str.contains('Tc1-IS630-Pogo', na=False) |
        repeat_df['attributes'].str.contains('En-Spm', na=False) |
        repeat_df['attributes'].str.contains('MULE-MuDR', na=False) |
        repeat_df['attributes'].str.contains('PiggyBac', na=False) |
        repeat_df['attributes'].str.contains('Tourist/Harbinger', na=False) |
        repeat_df['attributes'].str.contains('P-element', na=False) |
        repeat_df['attributes'].str.contains('Transib', na=False) |
        repeat_df['attributes'].str.contains('Mirage', na=False)
    )
    
    # Create dataframes for each category
    unknown_df = repeat_df[unknown_mask].copy()
    ltr_df = repeat_df[ltr_mask].copy()
    sine_df = repeat_df[sine_mask].copy()
    line_df = repeat_df[line_mask].copy()
    dna_transposon_df = repeat_df[dna_transposon_mask].copy()
    
    # Add category column to each dataframe
    unknown_df['repeat_category'] = 'Unknown'
    ltr_df['repeat_category'] = 'LTR'
    sine_df['repeat_category'] = 'SINE'
    line_df['repeat_category'] = 'LINE'
    dna_transposon_df['repeat_category'] = 'DNA_transposon'
    
    return {
        'Unknown': unknown_df,
        'LTR': ltr_df,
        'SINE': sine_df,
        'LINE': line_df,
        'DNA_transposon': dna_transposon_df
    }

# Apply the categorization
repeat_categories = categorize_repeats(repeats)

# Access individual dataframes
unknown_repeats = repeat_categories['Unknown']
ltr_repeats = repeat_categories['LTR']
sine_repeats = repeat_categories['SINE']
line_repeats = repeat_categories['LINE']
dna_transposon_repeats = repeat_categories['DNA_transposon']

# Print summary statistics
print("REPEAT CATEGORY SUMMARY")
print("="*60)
for category, df in repeat_categories.items():
    total_length = (df['End'] - df['Start']).sum()
    print(f"{category:<15} {len(df):>8,} repeats | {total_length:>12,} bp | {(total_length/1e6):>8.1f} Mbp")

print("\nOriginal total repeats:", len(repeats))
categorized_total = sum(len(df) for df in repeat_categories.values())
print("Categorized repeats:", categorized_total)
print("Uncategorized repeats:", len(repeats) - categorized_total)

In [ ]:
def categorize_repeats_exclusive(repeat_df):
    """
    Categorize RepeatMasker repeats into mutually exclusive categories
    using priority order to avoid double-counting
    """
    # Make a copy to avoid modifying original
    df = repeat_df.copy()
    
    # Initialize all as 'Other' first
    df['repeat_category'] = 'Other'
    
    # Define priority order (most specific to least specific)
    # Higher priority categories will claim repeats first
    
    # DNA transposons (most specific)
    dna_mask = (
        df['attributes'].str.contains('DNA', na=False) |
        df['attributes'].str.contains('hobo-Activator', na=False) |
        df['attributes'].str.contains('Tc1-IS630-Pogo', na=False) |
        df['attributes'].str.contains('En-Spm', na=False) |
        df['attributes'].str.contains('MULE-MuDR', na=False) |
        df['attributes'].str.contains('PiggyBac', na=False) |
        df['attributes'].str.contains('Tourist/Harbinger', na=False) |
        df['attributes'].str.contains('P-element', na=False) |
        df['attributes'].str.contains('Transib', na=False) |
        df['attributes'].str.contains('Mirage', na=False)
    )
    df.loc[dna_mask, 'repeat_category'] = 'DNA_transposon'
    
    # SINE elements
    sine_mask = (
        df['attributes'].str.contains('SINE', na=False) |
        df['attributes'].str.contains('Alu', na=False)
    ) & (df['repeat_category'] == 'Other')  # Only assign if not already categorized
    df.loc[sine_mask, 'repeat_category'] = 'SINE'
    
    # LINE elements
    line_mask = (
        df['attributes'].str.contains('LINE', na=False) |
        df['attributes'].str.contains('L1', na=False) |
        df['attributes'].str.contains('L2', na=False) |
        df['attributes'].str.contains('CR1', na=False) |
        df['attributes'].str.contains('Rex', na=False) |
        df['attributes'].str.contains('RTE', na=False) |
        df['attributes'].str.contains('CIN4', na=False)
    ) & (df['repeat_category'] == 'Other')
    df.loc[line_mask, 'repeat_category'] = 'LINE'
    
    # LTR elements
    ltr_mask = (
        df['attributes'].str.contains('LTR', na=False) |
        df['attributes'].str.contains('ltr-', na=False) |
        df['attributes'].str.contains('Retroviral', na=False) |
        df['attributes'].str.contains('Ty1/Copia', na=False) |
        df['attributes'].str.contains('Gypsy/DIRS1', na=False) |
        df['attributes'].str.contains('BEL/Pao', na=False)
    ) & (df['repeat_category'] == 'Other')
    df.loc[ltr_mask, 'repeat_category'] = 'LTR'
    
    # Unknown/Unclassified (lowest priority)
    unknown_mask = (
        df['attributes'].str.contains('Unknown', na=False) |
        df['attributes'].str.contains('Unclassified', na=False) |
        df['attributes'].str.contains('rnd-', na=False)
    ) & (df['repeat_category'] == 'Other')
    df.loc[unknown_mask, 'repeat_category'] = 'Unknown'
    
    return df

# Apply exclusive categorization
repeats_categorized = categorize_repeats_exclusive(repeats)

# Create separate dataframes
repeat_categories_exclusive = {}
for category in ['Unknown', 'LTR', 'SINE', 'LINE', 'DNA_transposon', 'Other']:
    repeat_categories_exclusive[category] = repeats_categorized[repeats_categorized['repeat_category'] == category].copy()

# Print corrected summary
print("EXCLUSIVE REPEAT CATEGORY SUMMARY")
print("="*70)
total_repeats = 0
total_length = 0

for category in ['Unknown', 'LTR', 'SINE', 'LINE', 'DNA_transposon', 'Other']:
    df = repeat_categories_exclusive[category]
    category_count = len(df)
    category_length = (df['End'] - df['Start']).sum()
    total_repeats += category_count
    total_length += category_length
    
    print(f"{category:<15} {category_count:>8,} repeats | {category_length:>12,} bp | {(category_length/1e6):>8.1f} Mbp | {(category_count/len(repeats)*100):>5.1f}%")

print("-"*70)
print(f"{'TOTAL':<15} {total_repeats:>8,} repeats | {total_length:>12,} bp | {(total_length/1e6):>8.1f} Mbp | {100:>5.1f}%")
print(f"Original total: {len(repeats):,} repeats")
print(f"Difference: {total_repeats - len(repeats):+,} repeats")

In [ ]:
repeats_categorized.columns

In [ ]:
## Filter segdup 
merged_segdup=merged_segdup[merged_segdup["chrom"].str.contains("chr")]

In [ ]:
def build_segdup_tree(segdup_df, chrom_col='chrom', start_col='start', end_col='end'):
    """Build interval tree for segdup regions (smaller dataset)"""
    trees = defaultdict(intervaltree.IntervalTree)
    for idx, row in segdup_df.iterrows():
        chrom = row[chrom_col]
        start = row[start_col]
        end = row[end_col]
        # Store the row index as data so we can map back results
        trees[chrom].addi(start, end, data=idx)
    return trees

def count_overlaps_optimized_with_progress(segdup_trees, target_df, segdup_total_count, 
                                         target_chrom_col='Chromosome', target_start_col='Start', target_end_col='End'):
    """
    Count overlaps by querying each target interval against segdup trees with progress bar
    """
    # Initialize counts for each segdup region (by index)
    counts = [0] * segdup_total_count
    
    # Create progress bar
    for _, target_row in tqdm(target_df.iterrows(), total=len(target_df), 
                            desc=f"Processing {len(target_df):,} intervals"):
        chrom = target_row[target_chrom_col]
        start = target_row[target_start_col]
        end = target_row[target_end_col]
        
        # Check if this target overlaps any segdup regions
        tree = segdup_trees.get(chrom, intervaltree.IntervalTree())
        overlapping_segdups = tree.overlap(start, end)
        
        # Increment counts for each overlapping segdup region
        for segdup_interval in overlapping_segdups:
            segdup_idx = segdup_interval.data
            if segdup_idx < segdup_total_count:  # Safety check
                counts[segdup_idx] += 1
    
    return counts

# Usage with progress bars
segdup_trees = build_segdup_tree(merged_segdup)

print("Counting gene overlaps...")
merged_segdup['gene_count'] = count_overlaps_optimized_with_progress(segdup_trees, genes, len(merged_segdup))

print("Counting repeat overlaps...")
merged_segdup['repeat_count'] = count_overlaps_optimized_with_progress(segdup_trees, repeats, len(merged_segdup))

In [ ]:
repeats_categorized['repeat_category'].unique()

In [ ]:
unknown_repeats = repeats_categorized[repeats_categorized['repeat_category']=="Unknown"]
LINE_repeats = repeats_categorized[repeats_categorized['repeat_category']=="LINE"]
LTR_repeats = repeats_categorized[repeats_categorized['repeat_category']=="LTR"]
SINE_repeats = repeats_categorized[repeats_categorized['repeat_category']=="SINE"]
DNA_transposon_repeats = repeats_categorized[repeats_categorized['repeat_category']=="DNA_transposon"]

In [ ]:
print("Counting unknown repeat overlaps...")
merged_segdup['unknown_repeat_count'] = count_overlaps_optimized_with_progress(segdup_trees, unknown_repeats, len(merged_segdup))

In [ ]:
print("Counting LINE repeat overlaps...")
merged_segdup['LINE_repeat_count'] = count_overlaps_optimized_with_progress(segdup_trees, LINE_repeats, len(merged_segdup))

In [ ]:
print("Counting LTR repeat overlaps...")
merged_segdup['LTR_repeat_count'] = count_overlaps_optimized_with_progress(segdup_trees, LTR_repeats, len(merged_segdup))

In [ ]:
print("Counting SINE repeat overlaps...")
merged_segdup['SINE_repeat_count'] = count_overlaps_optimized_with_progress(segdup_trees, SINE_repeats, len(merged_segdup))

In [ ]:
print("Counting DNA_transposon repeat overlaps...")
merged_segdup['DNA_transposon_repeat_count'] = count_overlaps_optimized_with_progress(segdup_trees, DNA_transposon_repeats, len(merged_segdup))

In [ ]:
# Create the scatterplot
plt.figure(figsize=(10, 6))

# Plot each point with chromosome-specific color
for i, row in merged_segdup.iterrows():
    chrom = row['chrom']
    color = chr_colors.get(chrom, '#CCCCCC')  # Default gray if chromosome not in dict
    plt.scatter(row['Length'], row['gene_count'], 
                color=color, alpha=0.7, s=50, label=chrom if i == 0 else "")

plt.xlabel('Segdup Region Length (bp)', fontsize=12)
plt.ylabel('Number of Overlapping Genes', fontsize=12)
plt.grid(True, alpha=0.3)

# Remove spines for publication style
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Optional: Log scale for x-axis if lengths vary widely
if merged_segdup['Length'].max() / merged_segdup['Length'].min() > 100:
    plt.xscale('log')
    plt.xlabel('Segdup Region Length (bp, log scale)', fontsize=12)

plt.tight_layout()
plt.show()

# Print correlation statistics
correlation = merged_segdup['Length'].corr(merged_segdup['gene_count'])
print(f"Correlation between length and gene count: {correlation:.3f}")

# Show some statistics
print(f"\nStatistics:")
print(f"Average length: {merged_segdup['Length'].mean():,.0f} bp")
print(f"Average gene count: {merged_segdup['gene_count'].mean():.2f}")
print(f"Largest region: {merged_segdup['Length'].max():,} bp with {merged_segdup.loc[merged_segdup['Length'].idxmax(), 'gene_count']} genes")
print(f"Region with most genes: {merged_segdup['gene_count'].max()} genes with length {merged_segdup.loc[merged_segdup['gene_count'].idxmax(), 'Length']:,} bp")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(merged_segdup['Length'], merged_segdup['gene_count'], alpha=0.5)
plt.xlabel('Length (bp)')
plt.ylabel('Gene Count')
plt.xscale("log")
plt.show()

In [ ]:
# Sort by Length for cumulative length
sorted_by_length = merged_segdup.sort_values('Length', ascending=False)
sorted_by_length['Cumulative_Length'] = sorted_by_length['Length'].cumsum()
total_length = sorted_by_length['Length'].sum()
sorted_by_length['Cumulative_Length_Pct'] = (sorted_by_length['Cumulative_Length'] / total_length) * 100

# Use the same sorted order for gene count calculations
sorted_by_length['Cumulative_Genes'] = sorted_by_length['gene_count'].cumsum()
total_genes = sorted_by_length['gene_count'].sum()
sorted_by_length['Cumulative_Genes_Pct'] = (sorted_by_length['Cumulative_Genes'] / total_genes) * 100

# Use the same sorted order for repeat count calculations
sorted_by_length['Cumulative_Repeats'] = sorted_by_length['repeat_count'].cumsum()
total_repeats = sorted_by_length['repeat_count'].sum()
sorted_by_length['Cumulative_Repeat_Pct'] = (sorted_by_length['Cumulative_Repeats'] / total_repeats) * 100

# Use the same sorted order for repeat count calculations
sorted_by_length['Cumulative_LINE'] = sorted_by_length['LINE_repeat_count'].cumsum()
total_repeats = sorted_by_length['LINE_repeat_count'].sum()
sorted_by_length['Cumulative_LINE_Pct'] = (sorted_by_length['Cumulative_LINE'] / total_repeats) * 100
# Use the same sorted order for repeat count calculations
sorted_by_length['Cumulative_Unknown'] = sorted_by_length['unknown_repeat_count'].cumsum()
total_repeats = sorted_by_length['unknown_repeat_count'].sum()
sorted_by_length['Cumulative_Unknown_Pct'] = (sorted_by_length['Cumulative_Unknown'] / total_repeats) * 100
# Use the same sorted order for repeat count calculations
sorted_by_length['Cumulative_SINE'] = sorted_by_length['SINE_repeat_count'].cumsum()
total_repeats = sorted_by_length['SINE_repeat_count'].sum()
sorted_by_length['Cumulative_SINE_Pct'] = (sorted_by_length['Cumulative_SINE'] / total_repeats) * 100
# Use the same sorted order for repeat count calculations
sorted_by_length['Cumulative_DNA_transposon'] = sorted_by_length['DNA_transposon_repeat_count'].cumsum()
total_repeats = sorted_by_length['DNA_transposon_repeat_count'].sum()
sorted_by_length['Cumulative_DNA_transposon_Pct'] = (sorted_by_length['Cumulative_DNA_transposon'] / total_repeats) * 100

# Create the plot
plt.figure(figsize=(10, 6))

# Cumulative Length
plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Cumulative_Length_Pct'], 
         label='Cumulative Length', linewidth=2, color="black")

# Cumulative Gene Count
plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Cumulative_Genes_Pct'], 
         label='Cumulative Gene Count', linewidth=2,alpha=0.3)

# Cumulative Repeat Count
plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Cumulative_Repeat_Pct'], 
         label='Cumulative Repeat Count', linewidth=2,alpha=0.3)

# Cumulative Repeat Count
plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Cumulative_LINE_Pct'], 
         label='Cumulative LINE Repeat Count', linewidth=2,alpha=0.3)
# Cumulative Repeat Count
plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Cumulative_Unknown_Pct'], 
         label='Cumulative Unknown Repeat Count', linewidth=2,alpha=0.3)
# Cumulative Repeat Count
plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Cumulative_SINE_Pct'], 
         label='Cumulative SINE Repeat Count', linewidth=2,alpha=0.3)
# Cumulative Repeat Count
plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Cumulative_DNA_transposon_Pct'], 
         label='Cumulative DNA Transposon Repeat Count', linewidth=2,alpha=0.3)

plt.xlabel('Total Segdup Regions sorted by Length (longest to shortest)')
plt.ylabel('Cumulative Percentage (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
curves={'Length': 'Cumulative_Length_Pct',
 'Genes': 'Cumulative_Genes_Pct',
 'All_Repeats': 'Cumulative_Repeat_Pct',
 'LINE': 'Cumulative_LINE_Pct',
 'Unknown': 'Cumulative_Unknown_Pct',
 'SINE': 'Cumulative_SINE_Pct',
 'DNA_transposon': 'Cumulative_DNA_transposon_Pct'}
curves

In [ ]:
# Analyze slopes for all cumulative curves
slope_analysis = {}

for name, col in curves.items():
    y = sorted_by_length[col].values
    slopes = np.diff(y)
    
    # Find where slope is closest to 1
    slope_differences = np.abs(slopes - 1.0)
    min_slope_idx = np.argmin(slope_differences)
    
    slope_1_region = min_slope_idx + 1
    slope_1_percentage = y[slope_1_region]
    slope_1_actual_slope = slopes[min_slope_idx]
    
    slope_analysis[name] = {
        'region': slope_1_region,
        'percentage': slope_1_percentage,
        'actual_slope': slope_1_actual_slope,
        'regions_percentage': (slope_1_region / len(sorted_by_length)) * 100
    }

# Print slope analysis
print("SLOPE = 1.0 ANALYSIS")
print("="*80)
print(f"{'Feature':<20} {'Region':<10} {'Cumulative %':<15} {'Actual Slope':<15} {'% of Regions':<15}")
print("-"*80)

for name, data in slope_analysis.items():
    print(f"{name:<20} {data['region']:<10,} {data['percentage']:<15.2f} {data['actual_slope']:<15.3f} {data['regions_percentage']:<15.1f}")

In [ ]:
size=17
# Create the plot with 80% threshold for Length only
plt.figure(figsize=(10, 6))

# Colors for different curves
colors = {
    'Length': 'black',
    'Genes': 'blue',
    'All_Repeats': 'red', 
    'LINE': 'orange',
    'Unknown': 'green',
    'SINE': 'purple',
    'DNA_transposon': 'brown'
}

# Plot each curve
for name, col in curves.items():
    color = colors[name]
    plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length[col], 
             label=name.replace('_', ' '), linewidth=2, 
             alpha=0.3 if name != 'Length' else 1.0,
             color=color)

# Mark 80% threshold for Length only
y_length = sorted_by_length['Cumulative_Length_Pct'].values
region_80_idx = np.argmax(y_length >= 80)
if y_length[region_80_idx] >= 80:
    plt.axvline(x=region_80_idx + 1, color='black', linestyle='--', alpha=0.2, linewidth=2)
    plt.plot(region_80_idx + 1, 80, 'o', color='black', markersize=8, 
            markerfacecolor='black', markeredgecolor='black', markeredgewidth=2,
            label=f'80% of cumulative length\nat {region_80_idx :,} region ')

# Add the 80% horizontal line
# plt.axhline(y=80, color='gray', linestyle='-', alpha=0.5, linewidth=1)

plt.xlabel('Total Segdup Regions sorted by Length (longest to shortest)',fontsize=size+2)
plt.ylabel('Cumulative Percentage (%)',fontsize=size+2)
plt.yticks(fontsize=size-2)
plt.xticks(fontsize=size-2)
# plt.title('Cumulative Distribution with 80% Length Threshold',fontsize=size)
plt.legend(title="Segdup Region Features",title_fontsize=size,bbox_to_anchor=(0.63, 0.60), loc='upper left',fontsize=13)
plt.grid(True, alpha=0.3)

# Remove spines for publication style
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
print(f"Correct 80% threshold: {correct_80_region:,} regions contain {correct_80_percentage:.2f}% of total length")

In [ ]:
segdup_hotspots = sorted_by_length[sorted_by_length["Cumulative_Length_Pct"]<=80]
segdup_hotspots = segdup_hotspots.rename(columns={'chrom': 'Chromosome'})
segdup_hotspots = segdup_hotspots.rename(columns={'start': 'Start'})
segdup_hotspots = segdup_hotspots.rename(columns={'end': 'End'})

segdup_hotspots.to_csv("segdup_hotspots_80percLength.tsv",sep="\t")

In [ ]:
print("This is how long the segdup regionsare are:")
sorted_by_length["Length"].sum()

In [ ]:
print("This is how long the segdup hotspots are:")
segdup_hotspots["Length"].sum()

In [ ]:
print(sorted_by_length[sorted_by_length["Cumulative_Length_Pct"]<=80]["Cumulative_Length_Pct"])

In [ ]:
# Cumulative Length
plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Length'], 
         label='Cumulative Length', linewidth=2, )
plt.show()
# plt.yscale("log")

# Cumulative Length
plt.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Length'], 
         label='Cumulative Length', linewidth=2, )
plt.yscale("log")
plt.show()

In [ ]:
# Create scatter plots with trend lines
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, (col_name, title) in enumerate(repeat_types):
    if i < len(axes):
        # Scatter plot
        scatter = axes[i].scatter(merged_segdup['Length'], merged_segdup[col_name], 
                                 alpha=0.5, s=10, c=merged_segdup[col_name], cmap='viridis')
        
        # # Add trend line
        # if len(merged_segdup) > 1:
        #     # z = np.polyfit(merged_segdup['Length'], merged_segdup[col_name], 1)
        #     # p = np.poly1d(z)
        #     axes[i].plot(merged_segdup['Length'], p(merged_segdup['Length']), 
        #                 "r--", alpha=0.8, linewidth=2)
        
        axes[i].set_xlabel('Segdup Length (bp)')
        axes[i].set_ylabel(f'Number of {title}')
        axes[i].set_title(f'{title} vs Segdup Length\nCorrelation: {merged_segdup["Length"].corr(merged_segdup[col_name]):.3f}')
        
        # Use log scale if data spans large range
        if merged_segdup['Length'].max() / merged_segdup['Length'].min() > 100:
            axes[i].set_xscale('log')
            axes[i].set_xlabel('Segdup Length (bp, log scale)')

plt.tight_layout()
plt.show()

In [ ]:
merged_segdup["gene_count"].value_counts()

In [ ]:
# Sort by Length for cumulative length
sorted_by_length = merged_segdup.sort_values('Length', ascending=False)
sorted_by_length['Cumulative_Length'] = sorted_by_length['Length'].cumsum()
total_length = sorted_by_length['Length'].sum()

# Sort by Gene Count for cumulative gene count  
sorted_by_genes = merged_segdup.sort_values('gene_count', ascending=False)
sorted_by_genes['Cumulative_Genes'] = sorted_by_genes['gene_count'].cumsum()
total_genes = sorted_by_genes['gene_count'].sum()

# Create the plot with two y-axes
fig, ax1 = plt.subplots(figsize=(10, 6))

# Cumulative Length on left y-axis
ax1.plot(range(1, len(sorted_by_length) + 1), sorted_by_length['Cumulative_Length'], 
         color='blue', label='Cumulative Length', linewidth=2)
ax1.set_xlabel('Number of Segdup Regions (sorted by respective metric)')
ax1.set_ylabel('Cumulative Length (bp)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Cumulative Gene Count on right y-axis
ax2 = ax1.twinx()
ax2.plot(range(1, len(sorted_by_genes) + 1), sorted_by_genes['Cumulative_Genes'], 
         color='red', label='Cumulative Gene Count', linewidth=2)
ax2.set_ylabel('Cumulative Gene Count', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.grid(True, alpha=0.3)
plt.show()